# Orpheus Live — GPU TTS server on Colab

Runs the **Orpheus TTS server** on this Colab GPU and exposes it through a free
Cloudflare tunnel. Your Mac keeps doing mic/STT/LLM locally and offloads only
synthesis here — the latency bottleneck.

**Before running:** `Runtime → Change runtime type → GPU` (T4 works; A100/L4 are faster).

Run every cell top to bottom. The last cell prints the exact command to run on your Mac.

In [ ]:
# 1) Confirm there's a GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU! Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2) Install server dependencies.
# vLLM is required — it's what makes a 3B model hit realtime on a T4 (plain
# transformers runs at ~0.1x realtime, so streamed audio underruns).
#
# Two traps, both solved by this ONE resolver invocation:
#  - CUDA-build mismatch: Colab preinstalls cu12x torch/torchvision/torchaudio
#    whose version numbers satisfy the pins, but the vLLM wheel links CUDA 13
#    (`libcudart.so.13` ImportError). --torch-backend=auto + --reinstall-package
#    force driver-matched builds.
#  - torch drift: installing torchvision in a SEPARATE command lets the resolver
#    upgrade torch past vLLM's exact pin, which breaks vLLM's compiled
#    flash-attn extensions (`_vllm_fa2_C` ImportError). Co-resolving with a
#    floor on vLLM makes the resolver downgrade torchvision instead of either
#    backtracking vLLM to ancient source-built releases or bumping torch.
%pip install -q uv
!uv pip install --system -q --torch-backend=auto \
    --reinstall-package torch --reinstall-package torchvision --reinstall-package torchaudio \
    "vllm>0.10" torchvision torchaudio fastapi uvicorn snac

# Fail fast (~15s) instead of at server launch: importing the llama model file
# exercises the compiled flash-attn extensions, the exact chain that breaks when
# torch's build drifts from vLLM's pin.
!python -c "import vllm.model_executor.models.llama; import torch, torchvision; print('OK: torch', torch.__version__, '| torchvision', torchvision.__version__)"

In [ ]:
# 3) Get the server file.
# Cloned from your repo automatically — no upload needed. Override REPO_URL only
# if you're working from a fork.
import os

REPO_URL = "https://github.com/andrewlau624/orpheus-live"

if REPO_URL:
    !rm -rf orpheus-live && git clone --depth 1 $REPO_URL orpheus-live
    SERVER = "orpheus-live/server/orpheus_server.py"
else:
    from google.colab import files
    print("Upload server/orpheus_server.py from your orpheus-live repo:")
    up = files.upload()
    SERVER = next(iter(up))

assert os.path.exists(SERVER), SERVER
print("Server file:", SERVER)

In [ ]:
# 4) Hugging Face access — the official Orpheus repo is GATED.
# Preferred: accept the license at https://huggingface.co/canopylabs/orpheus-3b-0.1-ft
# then add your HF token as a Colab secret named HF_TOKEN (key icon in the left
# sidebar, turn 'Notebook access' on). No secret? This cell falls back to an
# ungated mirror of the same weights (unsloth/orpheus-3b-0.1-ft).
import os

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None

if token:
    os.environ["HF_TOKEN"] = token
    print("HF token loaded from Colab secrets — using canopylabs/orpheus-3b-0.1-ft.")
else:
    os.environ["ORPHEUS_MODEL"] = "unsloth/orpheus-3b-0.1-ft"
    print("No HF_TOKEN secret found — using ungated mirror unsloth/orpheus-3b-0.1-ft.")

In [ ]:
# 5) Start the server in the background.
# First launch downloads the Orpheus model (~6 GB) — expect several minutes.
# Safe to re-run: kills any server left over from a previous attempt first,
# so you never health-check a stale process running old code. vLLM runs its
# engine in a CHILD process whose cmdline doesn't say "orpheus_server" — a
# stale one silently holds ~13GB of GPU memory, so also kill anything still
# registered on the GPU (the notebook kernel holds no CUDA context, so it
# isn't affected).
import subprocess, sys, time, urllib.request

!pkill -9 -f orpheus_server || true
!pkill -9 -f EngineCore || true
!nvidia-smi --query-compute-apps=pid --format=csv,noheader | xargs -r kill -9 2>/dev/null || true
!fuser -k 8000/tcp 2>/dev/null || true
time.sleep(3)

proc = subprocess.Popen(
    [sys.executable, SERVER],  # inherits os.environ, incl. HF_TOKEN / ORPHEUS_MODEL
    stdout=open("server.log", "w"),
    stderr=subprocess.STDOUT,
)

print("Waiting for the server (model download on first run — several minutes)...")
for _ in range(600):
    try:
        with urllib.request.urlopen("http://localhost:8000/health", timeout=2) as r:
            print("Server ready:", r.read().decode())
            break
    except Exception:
        if proc.poll() is not None:
            print(open("server.log").read()[-3000:])
            raise RuntimeError("server crashed — log above")
        time.sleep(2)
else:
    raise TimeoutError("server never came up — check server.log")

In [ ]:
# 6) Sanity check: synthesize a line and play it right here.
# vLLM on a T4 should hit ~1-3x realtime for smooth streaming.
import json, time, urllib.error, urllib.request
import numpy as np
from IPython.display import Audio, display

req = urllib.request.Request(
    "http://localhost:8000/tts",
    data=json.dumps({"text": "Hey! I'm running on a Colab GPU now.", "voice": "tara"}).encode(),
    headers={"Content-Type": "application/json"},
)
t0 = time.time()
try:
    raw = urllib.request.urlopen(req).read()
except urllib.error.HTTPError as e:
    print("TTS failed:", e.code, e.read().decode()[:2000])
    print("--- server.log tail ---")
    print(open("server.log").read()[-3000:])
    raise
pcm = np.frombuffer(raw[: len(raw) // 4 * 4], dtype="<f4")
secs = len(pcm) / 24000
took = time.time() - t0
print(f"{secs:.1f}s of audio in {took:.1f}s ({secs/took:.2f}x realtime)")
assert secs > 0.5, "got (almost) no audio — check server.log"
display(Audio(pcm, rate=24000))

In [ ]:
# 7) Open a public tunnel (free, no account) and print the Mac command
import re, subprocess, time

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!pkill -f cloudflared || true
time.sleep(1)

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
deadline = time.time() + 60
while time.time() < deadline and url is None:
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m:
        url = m.group(0)
assert url, "tunnel URL never appeared — rerun this cell"

print("Tunnel is up! On your Mac, run:\n")
print(f"  ORPHEUS_LIVE_TTS_BACKEND=remote \\\n  ORPHEUS_LIVE_TTS_REMOTE_URL={url} \\\n  make run")
print("\nKeep this notebook running for the whole conversation.")

### Notes

- **Keep this tab open** — Colab kills idle runtimes; the free tier disconnects after ~90 min idle.
- The tunnel URL changes every session; rerun cell 7 and update the env var on the Mac.
- First request after startup is slower (CUDA warmup + vLLM engine init); it settles after one or two sentences.
- Watch the `x realtime` number in cell 6. **vLLM on a T4 should hit ~1-3x realtime**, so streaming is smooth. On faster GPUs (L4/A100) it's even lower latency.
- Something broken? The server log is in `server.log` (`!tail -50 server.log`).